# Loading MAE  
My image: 224 x 224  
Patch size: 16 x 16  So: 224 / 16 = 14 patches per side  
Total patches: 14×14=196  

Then MAE normally masks 75%.  75% of 196: 0.75×196≈147  
Visible patches: 196−147=49  
Then add CLS token: 49+1=50  

outputs.hidden_states
→ encoder outputs before the final LayerNorm.
outputs.last_hidden_state
→ output after the final LayerNorm.

In [9]:
import torch
from transformers import AutoImageProcessor, ViTMAEModel #the encoder backbone only
from PIL import Image
import requests

#ViTMAEForPreTraining. This is the FULL MAE training pipeline: masking, reconstruction, decoder, loss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "facebook/vit-mae-large"

processor = AutoImageProcessor.from_pretrained(model_name)
model = ViTMAEModel.from_pretrained(model_name).to(device)
model.eval()

#disable masking
model.config.mask_ratio = 0.0

# test image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# preprocess
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
print("last_hidden_state:", last_hidden_state.shape)  # should be (1, 197, 1024)

cls_embedding = last_hidden_state[:, 0, :]
mean_patch_embedding = last_hidden_state[:, 1:, :].mean(dim=1)

print("cls:", cls_embedding.shape)          # (1, 1024)
print("mean pooled:", mean_patch_embedding.shape)  # (1, 1024)

C:\Users\louai\net2brain-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\louai\net2brain-env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


last_hidden_state: torch.Size([1, 197, 1024])
cls: torch.Size([1, 1024])
mean pooled: torch.Size([1, 1024])


In [12]:
activations = {}

def hook_fn(name):
    def hook(module, input, output):
        activations[name] = output
    return hook

model.layernorm.register_forward_hook(
    hook_fn("final_layernorm")
)

outputs = model(
    preprocessed_data,
    output_hidden_states=True
)

print(torch.allclose(
    outputs.last_hidden_state,
    activations["final_layernorm"]
))

NameError: name 'preprocessed_data' is not defined

In [10]:
print(outputs.keys())
print(num_patches)
print(inputs['pixel_values'].shape)
print(outputs['mask'].shape)
outputs.last_hidden_state.shape

odict_keys(['last_hidden_state', 'mask', 'ids_restore'])


NameError: name 'num_patches' is not defined

# DINOv2 not trained on imagenet1k  
DINOv2-large gives you:  
ViT-L scale (~300M params)  
stable/public HuggingFace support  
transformer SSL backbone  
widely accepted strong baseline  
much closer scale to MAE-L  
more scientifically defensible than DINOv3   

In [4]:
from transformers import AutoModel, AutoImageProcessor
from PIL import Image
import requests
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/dinov2-large"

processor = AutoImageProcessor.from_pretrained(model_name)

model = AutoModel.from_pretrained(model_name)
model = model.to(device)
model.eval()

# test image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# preprocess
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.keys())
print(outputs.last_hidden_state.shape)

odict_keys(['last_hidden_state', 'pooler_output'])
torch.Size([1, 257, 1024])


# Dinov1 trained on imagenet1k


In [1]:
import torch

dino = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_vitb16"
)

#print(hasattr(dino, "blocks"))
#print(len(dino.blocks))
#print(type(dino.blocks[0]))

Using cache found in C:\Users\louai/.cache\torch\hub\facebookresearch_dino_main


RuntimeError: Cannot find callable dino_vitl16 in hubconf

hidden_states[0] is usually: embeddings before any transformer block.  
hidden_states[1] is after block 1,  
hidden_states[-1]: is after the last block before final LayerNorm.  
For DINO, representations after each transformer block were extracted, resulting in 12 layers. For JEPA and MAE, the initial embedding representation provided by HuggingFace was excluded to maintain comparable transformer-depth analyses.

In [17]:
#testing if it has hidden states
import torch

# one test image
x = torch.randn(1, 3, 224, 224)

if torch.cuda.is_available():
    x = x.cuda()
    dino = dino.cuda()

features = {}

with torch.no_grad():
    tokens = dino.prepare_tokens(x)

    for i, block in enumerate(dino.blocks):
        tokens = block(tokens)

        pooled = tokens[:, 1:, :].mean(dim=1)

        features[f"layer_{i+1}"] = pooled

print(features.keys())

for k, v in features.items():
    print(k, v.shape)

dict_keys(['layer_1', 'layer_2', 'layer_3', 'layer_4', 'layer_5', 'layer_6', 'layer_7', 'layer_8', 'layer_9', 'layer_10', 'layer_11', 'layer_12'])
layer_1 torch.Size([1, 768])
layer_2 torch.Size([1, 768])
layer_3 torch.Size([1, 768])
layer_4 torch.Size([1, 768])
layer_5 torch.Size([1, 768])
layer_6 torch.Size([1, 768])
layer_7 torch.Size([1, 768])
layer_8 torch.Size([1, 768])
layer_9 torch.Size([1, 768])
layer_10 torch.Size([1, 768])
layer_11 torch.Size([1, 768])
layer_12 torch.Size([1, 768])


# IBOT  

| Property           | Value                    |
| ------------------ | ------------------------ |
| Dataset            | ImageNet-1K              |
| Model              | ViT-L/16                 |
| Parameters         | ~307M                    |
| Embedding dim      | 1024                     |
| Patch size         | 16                       |
| Image size         | 224                      |
| Tokens             | 197 (=196 patches + CLS) |
| Transformer blocks | 24                       |
| Checkpoint format  | Clean `state_dict`       |  

vit_base:  
embed_dim=768   
depth=12  
num_heads=12  
mlp_ratio=4  
qkv_bias=True  
norm_layer=partial(nn.LayerNorm, eps=1e-6)  


In [30]:
ckpt = torch.load(r'C:\Users\louai\Net2Brain\notebooks\ibot repo\ibot', map_location="cpu")

print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

<class 'dict'>
dict_keys(['state_dict'])


In [31]:
sd = ckpt["state_dict"]

print(len(sd))

for i, k in enumerate(sd.keys()):
    print(k)
    if i >= 30:
        break

294
cls_token
pos_embed
patch_embed.proj.weight
patch_embed.proj.bias
blocks.0.norm1.weight
blocks.0.norm1.bias
blocks.0.attn.qkv.weight
blocks.0.attn.qkv.bias
blocks.0.attn.proj.weight
blocks.0.attn.proj.bias
blocks.0.norm2.weight
blocks.0.norm2.bias
blocks.0.mlp.fc1.weight
blocks.0.mlp.fc1.bias
blocks.0.mlp.fc2.weight
blocks.0.mlp.fc2.bias
blocks.1.norm1.weight
blocks.1.norm1.bias
blocks.1.attn.qkv.weight
blocks.1.attn.qkv.bias
blocks.1.attn.proj.weight
blocks.1.attn.proj.bias
blocks.1.norm2.weight
blocks.1.norm2.bias
blocks.1.mlp.fc1.weight
blocks.1.mlp.fc1.bias
blocks.1.mlp.fc2.weight
blocks.1.mlp.fc2.bias
blocks.2.norm1.weight
blocks.2.norm1.bias
blocks.2.attn.qkv.weight


In [39]:
from functools import partial
import torch.nn as nn

ibot_model = vision_transformer.VisionTransformer(
    patch_size=16,
    embed_dim=1024,
    depth=24,
    num_heads=16,
    mlp_ratio=4,
    qkv_bias=True,
    norm_layer=partial(nn.LayerNorm, eps=1e-6),
)

msg = ibot_model.load_state_dict(sd, strict=True)

print(msg)

<All keys matched successfully>


In [40]:
# verify we can extract all layers.
ibot_model.eval()

x = torch.randn(1, 3, 224, 224)

if torch.cuda.is_available():
    x = x.cuda()
    ibot_model = ibot_model.cuda()

features = {}

with torch.no_grad():
    tokens = ibot_model.prepare_tokens(x)

    for i, block in enumerate(ibot_model.blocks):
        tokens = block(tokens)

        pooled = tokens[:, 1:, :].mean(dim=1)

        features[f"layer_{i+1}"] = pooled

print(features.keys())

for k, v in features.items():
    print(k, v.shape)

dict_keys(['layer_1', 'layer_2', 'layer_3', 'layer_4', 'layer_5', 'layer_6', 'layer_7', 'layer_8', 'layer_9', 'layer_10', 'layer_11', 'layer_12', 'layer_13', 'layer_14', 'layer_15', 'layer_16', 'layer_17', 'layer_18', 'layer_19', 'layer_20', 'layer_21', 'layer_22', 'layer_23', 'layer_24'])
layer_1 torch.Size([1, 1024])
layer_2 torch.Size([1, 1024])
layer_3 torch.Size([1, 1024])
layer_4 torch.Size([1, 1024])
layer_5 torch.Size([1, 1024])
layer_6 torch.Size([1, 1024])
layer_7 torch.Size([1, 1024])
layer_8 torch.Size([1, 1024])
layer_9 torch.Size([1, 1024])
layer_10 torch.Size([1, 1024])
layer_11 torch.Size([1, 1024])
layer_12 torch.Size([1, 1024])
layer_13 torch.Size([1, 1024])
layer_14 torch.Size([1, 1024])
layer_15 torch.Size([1, 1024])
layer_16 torch.Size([1, 1024])
layer_17 torch.Size([1, 1024])
layer_18 torch.Size([1, 1024])
layer_19 torch.Size([1, 1024])
layer_20 torch.Size([1, 1024])
layer_21 torch.Size([1, 1024])
layer_22 torch.Size([1, 1024])
layer_23 torch.Size([1, 1024])
layer

In [38]:
"""import inspect

print(inspect.signature(vision_transformer.VisionTransformer))
print(inspect.getsource(vision_transformer.vit_base))
print(sd["cls_token"].shape)
print(sd["pos_embed"].shape)
print(sd["blocks.0.attn.qkv.weight"].shape)
print(sd["norm.weight"].shape)
# #Pure backbone only? (t,f,f)
print("norm.weight" in sd) 
print("head.weight" in sd)
print("head.bias" in sd)"""

(img_size=[224], patch_size=16, in_chans=3, num_classes=0, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, qkv_bias=False, qk_scale=None, drop_rate=0.0, attn_drop_rate=0.0, drop_path_rate=0.0, norm_layer=<class 'torch.nn.modules.normalization.LayerNorm'>, **kwargs)
def vit_base(patch_size=16, **kwargs):
    model = VisionTransformer(
        patch_size=patch_size, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4,
        qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), **kwargs)
    return model



# Feature extraction

In [22]:
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
#paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

In [41]:
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor
import torch


# =========================
# preprocessing
# =========================

def my_preprocessor(image, model_name, device):
    """
    Args:
        image (Union[Image.Image, List[Image.Image]]): A PIL Image or a list of PIL Images.
        model_name (str): The name of the model, used to determine specific preprocessing if necessary.
        device (str): The device to which the tensor should be transferred ('cuda' for GPU, 'cpu' for CPU).

    Returns:
        Union[torch.Tensor, List[torch.Tensor]]: The preprocessed image(s) as PyTorch tensor(s).
    """

    transforms = trn.Compose([
        trn.Resize((224, 224)),
        trn.ToTensor(),
        trn.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])

    img_tensor = transforms(image).unsqueeze(0)

    if device == 'cuda':
        img_tensor = img_tensor.cuda()

    return img_tensor


# =========================
# generic CNN extractor
# =========================

def my_extractor(preprocessed_data, layers_to_extract, model):

    extractor_model = tx.Extractor(model, layers_to_extract)

    _, features = extractor_model(preprocessed_data)

    return features


# =========================
# generic transformer extractor
# (JEPA / MAE / DINOv2)
# =========================

def my_transformer_extractor(preprocessed_data, layers_to_extract, model):

    outputs = model.model(
        preprocessed_data,
        output_hidden_states=True
    )

    hidden_states = outputs.hidden_states

    features = {}

    for i, layer in enumerate(hidden_states):

        # remove CLS token
        layer = layer[:, 1:, :]

        # mean-pool patch tokens
        layer = layer.mean(dim=1)

        features[f"layer_{i}"] = layer

    return features

# =========================
# DINOv1
# =========================

def my_dino_transformer_extractor(preprocessed_data, layers_to_extract, model):

    dino = model.model

    features = {}

    with torch.no_grad():
        tokens = dino.prepare_tokens(preprocessed_data)

        for i, block in enumerate(dino.blocks):
            tokens = block(tokens)

            pooled = tokens[:, 1:, :].mean(dim=1)

            features[f"layer_{i+1}"] = pooled

    return features

# =========================
# IBOT
# =========================

def my_ibot_transformer_extractor(preprocessed_data,
                                  layers_to_extract,
                                  model):

    ibot = model.model

    features = {}

    with torch.no_grad():
        tokens = ibot.prepare_tokens(preprocessed_data)

        for i, block in enumerate(ibot.blocks):
            tokens = block(tokens)

            pooled = tokens[:, 1:, :].mean(dim=1)

            features[f"layer_{i+1}"] = pooled

    return features

# =========================
# cleaner
# =========================

def my_cleaner(features):
    return features

In [42]:
import torch.nn as nn

class TransformerFeatureExtractor(nn.Module):

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):

        with torch.no_grad():
            outputs = self.model(pixel_values=x)

        return outputs
transformer_extractor = TransformerFeatureExtractor(model)

class DINOFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

dino_wrapper = DINOFeatureExtractor(dino)

class IBOTFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

ibot_wrapper = IBOTFeatureExtractor(ibot_model)

In [44]:
# ibot extractor
fx = FeatureExtractor(
    model=ibot_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_ibot_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path='IBOT_Feat_nsd'
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [23:32<00:00,  1.62s/it, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [01:19<00:00, 10.94it/s]


In [26]:
# DINOv1 extractor
fx = FeatureExtractor(
    model=dino_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_dino_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path=r'C:\Users\louai\Net2Brain\notebooks\0_FEATURE EXTRACTIONS\DINO_Feat_nsd'
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [07:20<00:00,  1.98it/s, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [01:15<00:00, 11.60it/s]


In [25]:
# add MAE extractor. iran this accidently for 1 sec
fx = FeatureExtractor(
    model=transformer_extractor,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_transformer_extractor
)
fx.extract(
    data_path=stimuli_path,
    save_path=r'C:\Users\louai\Net2Brain\notebooks\0_FEATURE EXTRACTIONS\MAE_Feat_nsd'
)

Processing files:   0%|                                                  | 2/872 [00:04<34:01,  2.35s/it, subfiles=1/1]


KeyboardInterrupt: 

In [ ]:
# DINO extractor
fx = FeatureExtractor(
    model=transformer_extractor,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_transformer_extractor
)
fx.extract(
    data_path=stimuli_path,
    save_path='DINO_Feat_nsd'
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [32:48<00:00,  2.26s/it, subfiles=1/1]


Consolidating data per layer...


 20%|████████████████▏                                                               | 176/872 [00:14<01:09,  9.99it/s]

# LeJEPA

In [1]:
!git clone https://github.com/galilai-group/lejepa.git

Cloning into 'lejepa'...


In [2]:
import os

os.listdir("lejepa")

['.git',
 'eval',
 'figures',
 'lejepa',
 'LICENSE',
 'MINIMAL.md',
 'pyproject.toml',
 'README.md',
 'scripts',
 'setup.py',
 'tests']

In [3]:
os.listdir("lejepa/lejepa")

['multivariate', 'univariate', '__init__.py', '__pycache__']

In [4]:
# Find where the pretrained ViT models are defined.
import os

for root, dirs, files in os.walk("lejepa"):
    for f in files:
        if f.endswith(".py"):
            path = os.path.join(root, f)

            with open(path, "r", encoding="utf-8") as file:
                text = file.read()

            if "vit" in text.lower() or "ViT" in text:
                print(path)

lejepa\lejepa\multivariate\comb.py
lejepa\lejepa\multivariate\hz.py
lejepa\lejepa\univariate\watson.py


In [5]:
with open("lejepa/README.md", "r", encoding="utf-8") as f:
    print(f.read()[:4000])

# LeJEPA
**Lean Joint-Embedding Predictive Architecture (LeJEPA): Provable and Scalable Self-Supervised Learning Without the Heuristics**
[GitHub Repository](https://github.com/rbalestr-lab/lejepa)  
[arXiv:2511.08544](https://arxiv.org/abs/2511.08544)
---

Rush to our [minimal working example](MINIMAL.md) to see a full-fledge working example (ViT, inet).

## Demo

<img src="eval/output1.gif" controls width="400">
<img src="eval/output2.gif" controls width="400">
<img src="eval/output3.gif" controls width="400">
<table>
  <tr>
    <td><img src="eval/n01818515_919_original.png" width="200"/></td>
    <td><img src="eval/n01818515_919_pca.png" width="200"/></td>
  </tr>
  <tr>
    <td><img src="eval/n01818515_14304_original.png" width="200"/></td>
    <td><img src="eval/n01818515_14304_pca.png" width="200"/></td>
  </tr>
</table>

| shots | model                  | params | pretrain | epochs | DTD      | aircr.   | cars     | cifar10  | cifar100 | flowers102 | food     | pets     | avg.  

In [6]:
with open("lejepa/README.md", "r", encoding="utf-8") as f:
    text = f.read()

keywords = [
    "checkpoint",
    "weights",
    "pretrained",
    "huggingface",
    "download",
    ".pth",
    ".pt"
]

for k in keywords:
    print("\n---", k, "---")
    print(k in text.lower())


--- checkpoint ---
False

--- weights ---
False

--- pretrained ---
False

--- huggingface ---
False

--- download ---
False

--- .pth ---
False

--- .pt ---
False


In [7]:
!git clone https://github.com/rbalestr-lab/lejepa.git lejepa_full

Cloning into 'lejepa_full'...


In [8]:
import os

os.listdir("lejepa_full")

['.git',
 'eval',
 'figures',
 'lejepa',
 'LICENSE',
 'MINIMAL.md',
 'pyproject.toml',
 'README.md',
 'scripts',
 'setup.py',
 'tests']

In [9]:
import os

os.listdir("lejepa/scripts")

['launch_epps_ablation.md',
 'launch_inet10.py',
 'launch_proj_ablation.md',
 'launch_rtokens_ablation.md',
 'launch_views_ablation.md',
 '__pycache__']

In [10]:
with open("lejepa/MINIMAL.md", "r", encoding="utf-8") as f:
    print(f.read()[:6000])

# LeJEPA: Minimal Example
*Author: Randall Balestriero, Yann LeCun*  
*Date: 2025-11-20*
---

This page explains a minimal implementation of [**LeJEPA**](https://arxiv.org/abs/2511.08544)  using a Vision Transformer (ViT) backbone and the [ImageNette](https://github.com/fastai/imagenette) dataset. The code builds upon the code snippets provided in the paper to provide a simplified version of the full method, designed for easy experimentation and extension, with the following core benefits:
- 130 lines total
- Wandb logging
- ViT and Imagenette (inet10)
- no teacher-student, stop-gradient, ...
- online linear probing
- SOTA top1 accuracy

Before reading more, make sure you have the minimal dependencies installed 
```bash
pip install torch torchvision timm wandb hydra-core datasets huggingface-hub
```
## Table of Contents
1. [Why LeJEPA?](#why-lejepa)
2. [The Code](#the-code)
3. [Case-Study: ViT/s-8 + Imagenette](#case-study-vits-8--imagenette)
4. [Reference](#reference)
---
## Why LeJEP

In [11]:
with open("lejepa/scripts/launch_inet10.py", "r", encoding="utf-8") as f:
    print(f.read()[:8000])

import stable_pretraining as spt

TEMPLATE = """
HYDRA_FULL_ERROR=1 python scripts/je.py --config-dir scripts/configs --config-name base +accelerator=single_gpu_sc ++bstat_name="epps_pulley" ++bstat_num_slices=1000 ++dataset_name="inet10" ++max_epochs=400 ++batch_size=512 ++bstat_lambda=0.01,0.02,0.05,0.1 ++embedding_dim=512 ++projector_dim=512 ++projector_arch="MLP" ++lr=3e-3,1e-4 ++weight_decay=3e-2,1e-5 ++num_nodes=1 ++hydra.launcher.gpus_per_node=2 ++teacher_student=false ++resolution=224 ++n_views=8 ++random_erasing_p=0 ++drop_path_rate=0 ++multi_crop=false ++predictor=false ++autostop=true ++wandb_project=null 
"""

names = []
count = 0
for name, value in spt.static.TIMM_PARAMETERS.items():
    if value < 20000000 and "." not in name:
        count += 1
        names.append(f'"{name}"')

print(f"RUN\n\n{TEMPLATE} ++backbone={','.join(names)}")



In [13]:
import os

for root, dirs, files in os.walk("lejepa"):
    for f in files:
        if f == "je.py":
            print(os.path.join(root, f))

In [14]:
import os

for root, dirs, files in os.walk("lejepa/scripts"):
    print(root)
    print(files)
    print()

lejepa/scripts
['launch_epps_ablation.md', 'launch_inet10.py', 'launch_proj_ablation.md', 'launch_rtokens_ablation.md', 'launch_views_ablation.md']

lejepa/scripts\__pycache__
['ssl.cpython-313.pyc']

